Task 1: Create Database


In [25]:
import sqlite3
import random
import pandas as pd

In [26]:
conn=sqlite3.connect('employees.db')
cursor=conn.cursor()

In [27]:
cursor.execute('''
    create table IF NOT EXISTS employees ( 
               emp_id INTEGER PRIMARY KEY ,
               name TEXT NOT NULL, 
               DEPARTMENT TEXT NOT NULL,
               SALARY REAL NOT NULL ,
               years_experience INTEGER NOT NULL,
               performance_score REAL NOT NULL
               )
               ''')


In [28]:

departments = ["Engineering", "Sales", "Marketing", "HR", "Finance"]

employees = []

# generate 40 records
for i in range(1, 41):
    name = f"Employee_{i}"
    department = random.choice(departments)
    salary = random.randint(50000, 150000)
    years_experience = random.randint(1, 15)
    performance_score = round(random.uniform(1.0, 5.0), 2)

    employees.append((i, name, department, salary, years_experience, performance_score))

# insert records
cursor.executemany("""
INSERT INTO employees (emp_id, name, department, salary, years_experience, performance_score)
VALUES (?, ?, ?, ?, ?, ?)
""", employees)

conn.commit()

print("employees.db created with 40 employee records.")

employees.db created with 40 employee records.


In [29]:
cursor.execute("select * from employees")
print(cursor.fetchall())

[(1, 'Employee_1', 'HR', 89747.0, 9, 3.03), (2, 'Employee_2', 'Marketing', 114468.0, 8, 2.12), (3, 'Employee_3', 'Finance', 114145.0, 1, 4.42), (4, 'Employee_4', 'HR', 149599.0, 5, 4.1), (5, 'Employee_5', 'Marketing', 102159.0, 6, 1.6), (6, 'Employee_6', 'Finance', 68822.0, 14, 2.3), (7, 'Employee_7', 'HR', 142620.0, 15, 3.62), (8, 'Employee_8', 'HR', 115679.0, 5, 4.01), (9, 'Employee_9', 'Sales', 139202.0, 3, 1.55), (10, 'Employee_10', 'Marketing', 112744.0, 8, 2.1), (11, 'Employee_11', 'Finance', 115436.0, 2, 2.98), (12, 'Employee_12', 'Engineering', 58016.0, 4, 1.03), (13, 'Employee_13', 'Finance', 112803.0, 6, 4.01), (14, 'Employee_14', 'HR', 68337.0, 14, 2.06), (15, 'Employee_15', 'Engineering', 123318.0, 7, 1.4), (16, 'Employee_16', 'Marketing', 129189.0, 5, 2.48), (17, 'Employee_17', 'HR', 59005.0, 9, 2.51), (18, 'Employee_18', 'Engineering', 89630.0, 9, 4.53), (19, 'Employee_19', 'Finance', 76795.0, 12, 1.3), (20, 'Employee_20', 'HR', 70769.0, 6, 2.14), (21, 'Employee_21', 'Fin

In [30]:
#cursor.execute("DROP TABLE IF EXISTS employees")

Task 2: SQL Queries


Query 1: Employees with performance_score >= 4.0 AND years_experience >= 3. Show name, department, salary, performance_score. Sort by performance_score DESC. Limit 15.



In [35]:
Q1='''select name,department,salary,performance_score from employees
where performance_score>=4.0 and years_experience>=3 
order by performance_score desc
limit 15'''
df1=pd.read_sql(Q1,conn)
df1


,name,DEPARTMENT,SALARY,performance_score
0,Employee_28,Marketing,128760.0,4.70
1,Employee_34,Finance,141428.0,4.64
2,Employee_18,Engineering,89630.0,4.53
3,Employee_40,HR,99221.0,4.53
4,Employee_37,Sales,102786.0,4.46
5,Employee_30,Sales,61256.0,4.12
6,Employee_4,HR,149599.0,4.10
7,Employee_22,Engineering,53456.0,4.07
8,Employee_8,HR,115679.0,4.01
9,Employee_13,Finance,112803.0,4.01


Query 2: Employees with salary BETWEEN 
70
,
000
A
N
D
70,000AND110,000 in Engineering OR Sales. Show all columns. Sort by department, then salary DESC.

In [44]:
q2 = '''
SELECT * 
FROM employees
WHERE salary BETWEEN 70000 AND 110000
AND department IN ('Engineering', 'Sales')
'''

df2 = pd.read_sql(q2, conn)
df2

,emp_id,name,DEPARTMENT,SALARY,years_experience,performance_score
0,18,Employee_18,Engineering,89630.0,9,4.53
1,37,Employee_37,Sales,102786.0,5,4.46
2,39,Employee_39,Engineering,82857.0,1,1.54


Query 3: Count employees per department and calculate average salary per department. Sort by avg_salary DESC.

In [53]:
q3='''select department,count(emp_id) as no_of_employees , avg(salary) as average_salary  from employees
group by department
order by average_salary desc'''
df3=pd.read_sql(q3,conn)
df3



,DEPARTMENT,no_of_employees,average_salary
0,Marketing,9,102441.555556
1,Finance,9,100287.000000
2,HR,12,96026.500000
3,Engineering,6,92680.500000
4,Sales,4,90795.250000


Task 3: Pandas Analysis
Calculate average performance_score by department using .groupby()


In [79]:
q4='select * from employees'
df4=pd.read_sql(q4,conn)
df4

,emp_id,name,DEPARTMENT,SALARY,years_experience,performance_score
0,1,Employee_1,HR,89747.0,9,3.03
1,2,Employee_2,Marketing,114468.0,8,2.12
2,3,Employee_3,Finance,114145.0,1,4.42
3,4,Employee_4,HR,149599.0,5,4.10
4,5,Employee_5,Marketing,102159.0,6,1.60
5,6,Employee_6,Finance,68822.0,14,2.30
6,7,Employee_7,HR,142620.0,15,3.62
7,8,Employee_8,HR,115679.0,5,4.01
8,9,Employee_9,Sales,139202.0,3,1.55
9,10,Employee_10,Marketing,112744.0,8,2.10


In [77]:
avg_performace_score= df4.groupby('DEPARTMENT')['performance_score'].mean()
avg_performace_score

DEPARTMENT
Engineering    2.418333
Finance        3.065556
HR             3.241667
Marketing      2.661111
Sales          2.925000
Name: performance_score, dtype: float64

Replicate Query 1 using ONLY Pandas (.loc[], .sort_values(), .head())


In [76]:
'''select name,department,salary,performance_score from employees
where performance_score>=4.0 and years_experience>=3 
order by performance_score desc
limit 15'''
q1_d1 = (
    df4.loc[(df4['performance_score'] >= 4.0) & (df4['years_experience'] >= 3)]
    [['name', 'DEPARTMENT', 'SALARY', 'performance_score']]
    .sort_values('performance_score', ascending=False)
    .head(15)
)
q1_d1

,name,DEPARTMENT,SALARY,performance_score
27,Employee_28,Marketing,128760.0,4.70
33,Employee_34,Finance,141428.0,4.64
17,Employee_18,Engineering,89630.0,4.53
39,Employee_40,HR,99221.0,4.53
36,Employee_37,Sales,102786.0,4.46
29,Employee_30,Sales,61256.0,4.12
3,Employee_4,HR,149599.0,4.10
21,Employee_22,Engineering,53456.0,4.07
7,Employee_8,HR,115679.0,4.01
12,Employee_13,Finance,112803.0,4.01



3.Write 150-200 words comparing SQL vs Pandas:
Key syntax differences (=, AND, ORDER BY vs ==, &, .sort_values())
When to use each tool
Which is better for datasets too large for memory? Why?**


SQL and Pandas are both used to work with data, but their syntax and use cases are different. In SQL, we query data using statements like SELECT, WHERE, AND, and ORDER BY. For example, SQL uses = to compare values and AND to combine conditions. In Pandas, which is a Python library, we use Python-style syntax. Comparisons use == instead of =, conditions are combined with & instead of AND, and sorting is done using .sort_values() instead of ORDER BY.

SQL is mainly used when data is stored in databases such as MySQL, PostgreSQL, or SQLite. It is very efficient for querying large structured datasets directly from the database. Pandas is usually used when working with data analysis in Python, such as cleaning data, performing calculations, or building machine learning models. It is especially useful when data is already loaded into memory and we want to manipulate it quickly.

For datasets that are too large to fit into memory, SQL is generally better because databases are designed to handle large data efficiently on disk. Pandas loads data into RAM, so if the dataset is extremely large, it may run out of memory. Therefore, SQL is preferred for very large datasets.